# 05 — Bottleneck Audit
Rung 05. Baseline = `02_lora_sft`.
**The ONE variable:** The image passed to the model (Real vs Black vs Shuffled).

This notebook implements the A/B test to see if the ViT is being used, or if the model relies purely on a language shortcut.

In [ ]:
import sys
from pathlib import Path

# Bootstrap: Walk up to the repo root
EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent

for p in (EXP_DIR / "_models", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

## Sibling rungs
- This notebook runs arms `a0_real`, `a1_black`, `a2_shuffled` directly.

In [ ]:
import time
import pandas as pd
from PIL import Image

from frame.config import BaselineConfig
from frame.data import load_frame_items, FrameProvider
from frame.engine import QwenFrameEngine
from frame.delta import correctness_frame, question_metadata
from focus.data.data_models import Response
from focus.enums import Track
from focus.evaluation.evaluator import Evaluator

# We import the arms from the ablation engine
from ablation import ARMS, build_shuffle_map, apply_arm, tensor_fingerprint

SMOKE = True  # Set to True for G2 and G2b tests (n=20)

cfg = BaselineConfig(
    model_path=Path("/workspace/repo/experiments/02-lora-sft/runs/02_lora_sft_v1/merged/checkpoint-1720"),
    out_dir=EXP_DIR / "runs" / "05_bottleneck_audit",
    n_eval=20 if SMOKE else None
)
cfg.out_dir.mkdir(parents=True, exist_ok=True)


## Step 0: Crosstab + Trivial Floor (T0)

In [ ]:
print("Running T0: Crosstab and trivial floor on TRAIN and VAL...")
train_items = load_frame_items(cfg, splits=("train",))
val_items = load_frame_items(cfg, splits=("test",))

def items_to_df(items):
    rows = []
    for it in items:
        req, ref = it.request, it.reference
        group = getattr(ref.primary, "group", ref.primary)
        group_name = getattr(group, "value", str(group))
        ans = ref.answer
        rows.append({
            "answer_format": getattr(ref, "_format", "unknown"),
            "capability_group": group_name,
            "answer": str(ans).lower() if ans else "",
        })
    return pd.DataFrame(rows)

df_train = items_to_df(train_items)
df_val = items_to_df(val_items)

ct_train = pd.crosstab(df_train['answer_format'], df_train['capability_group'])
print("--- CROSSTAB (TRAIN) ---")
print(ct_train)

ct_val = pd.crosstab(df_val['answer_format'], df_val['capability_group'])
print("\n--- CROSSTAB (VAL) ---")
print(ct_val)

ct_train.to_csv(cfg.out_dir / "t0_crosstab_train.csv")
ct_val.to_csv(cfg.out_dir / "t0_crosstab_val.csv")

trivial_acc_val_majority = {}
trivial_acc_train_prior = {}
print("\n--- TRIVIAL FLOORS (ACCURACY ON VAL) ---")
for fmt in df_val['answer_format'].unique():
    train_sub = df_train[df_train['answer_format'] == fmt]
    val_sub = df_val[df_val['answer_format'] == fmt]
    
    if not train_sub.empty and not val_sub.empty:
        mode_train = train_sub['answer'].mode().iloc[0]
        # 1) The REAL floor: the best possible blind strategy on val
        acc_val_maj = val_sub['answer'].value_counts(normalize=True).max()
        # 2) Informative: what it would get if it emits the train prior
        acc_train_prior = (val_sub['answer'] == mode_train).mean()
    else:
        acc_val_maj = 0.0
        acc_train_prior = 0.0
        
    trivial_acc_val_majority[fmt] = acc_val_maj
    trivial_acc_train_prior[fmt] = acc_train_prior
    print(f"Format: {fmt} | Majority (Val): {acc_val_maj:.4f} | Prior (Train): {acc_train_prior:.4f}")


## Test A: Inference per arm

In [ ]:
eval_items = load_frame_items(cfg, splits=("test",))
if cfg.n_eval:
    stride = max(1, len(eval_items) // cfg.n_eval)
    eval_items = eval_items[::stride][:cfg.n_eval]

print(f"Unique video_ids in eval_items: {len({it.video_id for it in eval_items})}")

shuffle_map = build_shuffle_map(eval_items, seed=cfg.seed)

# Materialize frames to disk to avoid saturating the FrameProvider (B4)
# Lossless PNG is used (C1)
frames_dir = cfg.out_dir / "frames"
frames_dir.mkdir(parents=True, exist_ok=True)
cache_items = sorted(eval_items, key=lambda it: (it.dataset, it.video_id, it.frame_index))

print(f"\nCaching {len(cache_items)} frames to {frames_dir} ...")
provider = FrameProvider(cfg)
for i, item in enumerate(cache_items):
    provider.ensure_reader(item)
    img = provider.get_frame(item)
    img.save(frames_dir / f"{item.request.qID}.png", "PNG")
    if (i + 1) % 100 == 0 or (i + 1) == len(cache_items):
        print(f"  Cached {i + 1}/{len(cache_items)}")
provider.close()

engine = QwenFrameEngine(cfg)

# Dictionaries to store outputs and fingerprints
F = {a: [] for a in ARMS}
OUT = {a: [] for a in ARMS}
responses_by_arm = {a: [] for a in ARMS}

for arm in ARMS:
    print(f"\n=== Running arm: {arm} ===")
    engine.load()
    t_start = time.time()
    
    for i, item in enumerate(eval_items):
        image = Image.open(frames_dir / f"{item.request.qID}.png").copy()
        if arm == "a2_shuffled":
            donor_qID = eval_items[shuffle_map[item.request.qID]].request.qID
            donor = Image.open(frames_dir / f"{donor_qID}.png").copy()
        else:
            donor = None
        image = apply_arm(image, arm, donor=donor)
        
        # G2 Fingerprint
        F[arm].append(tensor_fingerprint(image))
        
        content = engine.predict(image, item.request.question)
        OUT[arm].append(content)
        
        responses_by_arm[arm].append(Response(qID=item.request.qID, content=content, latency=0.0))
        
        if (i + 1) % 50 == 0 or i + 1 == len(eval_items):
            rate = (i + 1) / (time.time() - t_start)
            eta = (len(eval_items) - i - 1) / rate if rate else 0
            print(f"[{arm}] infer {i + 1}/{len(eval_items)}  {rate:.2f} q/s  ETA {eta/60:.0f} min")
            
    engine.unload()


## Gates G2 and G2b

In [ ]:
# G2: The 3 arms receive distinct tensors (C5: we verify all eval_items are processed)
n_check = len(eval_items)
for i in range(n_check):
    assert F["a0_real"][i] != F["a1_black"][i], f"item {i}: the BLACK ablation did not happen"
    assert F["a0_real"][i] != F["a2_shuffled"][i], f"item {i}: the SHUFFLE did not happen"
print("OK G2: the 3 arms receive distinct tensors")

# G2b: The ablation bites
diff = sum(1 for i in range(n_check) if OUT["a0_real"][i] != OUT["a1_black"][i])
print(f"G2b: {diff}/{n_check} answers change when the image is ablated")


## Evaluation and RESULTS.csv

In [ ]:
# G3: Integrity of the full run
ids = {a: {r.qID for r in responses_by_arm[a]} for a in ARMS}
assert ids["a0_real"] == ids["a1_black"] == ids["a2_shuffled"], "the arms do not share qIDs"
if not SMOKE:
    assert len(ids["a0_real"]) == 6252, f"missing items: {len(ids['a0_real'])}"
for a in ARMS:
    assert len(responses_by_arm[a]) == len({r.qID for r in responses_by_arm[a]}), f"{a}: duplicate qID -> SDK gate 4"

results_dfs = {}
requests = [it.request for it in eval_items]
references = [it.reference for it in eval_items]

from focus.evaluation.judges import TransformersJudge
judge = TransformersJudge(model_name=cfg.judge_model, device=cfg.device)
evaluator = Evaluator(judges=[judge], seed=cfg.seed)

for arm in ARMS:
    print(f"\nEvaluating arm: {arm}")
    results_df, summary_df = evaluator.run(
        requests=requests,
        references=references,
        responses=responses_by_arm[arm],
        output_dir=cfg.out_dir / arm,
        track=Track.FRAME
    )
    results_dfs[arm] = results_df
    
# Compilation of RESULTS.csv
split_df = pd.read_csv(REPO / "experiments/splits/frame_ood_v1.csv")
video_split = dict(zip(zip(split_df.dataset, split_df.video_id), split_df.split))
meta = question_metadata(eval_items, video_split=video_split)

results_rows = []
for arm in ARMS:
    correctness_df = correctness_frame(results_dfs[arm])
    m = meta.merge(correctness_df, on="qID")
    
    row_data = {"arm": arm}
    
    # overall
    row_data["acc_overall"] = m["correct"].mean()
    row_data["acc_ID"] = m[m.distribution == "ID"]["correct"].mean()
    row_data["acc_OOD"] = m[m.distribution == "OOD"]["correct"].mean()
    
    # by format
    for fmt in m["answer_format"].unique():
        row_data[f"acc_fmt_{fmt}"] = m[m["answer_format"] == fmt]["correct"].mean()
        
    # by bucket (4 official buckets)
    for grp in m["capability_group"].unique():
        for dist in ("ID", "OOD"):
            sel = m[(m.capability_group == grp) & (m.distribution == dist)]
            if len(sel) > 0:
                row_data[f"acc_bucket_{grp}_{dist}"] = sel["correct"].mean()
                row_data[f"n_bucket_{grp}_{dist}"] = len(sel)
                
    # bucket_mean (excluding temporal_grounding if it existed accidentally)
    valid_buckets = ["object_recognition", "aggregation"]
    bucket_cols = []
    for grp in valid_buckets:
        for dist in ("ID", "OOD"):
            col = f"acc_bucket_{grp}_{dist}"
            if col in row_data:
                bucket_cols.append(col)
                
    if bucket_cols:
        row_data["bucket_mean"] = sum(row_data[c] for c in bucket_cols) / len(bucket_cols)
    else:
        row_data["bucket_mean"] = float('nan')
        
    results_rows.append(row_data)
    
final_df = pd.DataFrame(results_rows)

# Attach the trivial floors (C3)
for fmt, val in trivial_acc_val_majority.items():
    final_df[f"trivial_floor_val_majority_{fmt}"] = val
for fmt, val in trivial_acc_train_prior.items():
    final_df[f"trivial_floor_train_prior_{fmt}"] = val
    
if not SMOKE:
    final_df.to_csv(REPO / "experiments/05-bottleneck-audit/RESULTS.csv", index=False)
    print("Wrote RESULTS.csv with per-bucket and per-format stats and trivial floors.")
else:
    print("SMOKE mode: skipped writing RESULTS.csv. DataFrame preview:")
    print(final_df)
